In [42]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem, DataStructs
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor

In [45]:
test_df = pd.read_csv(r"..\data\raw\cyp-challenge-TRAIN_TDI.csv")
print(test_df.shape[0])
test_df.head()

6145


,Molecule_Name,SMILES,CYP2D6_is_TDI,CYP3A4_is_TDI,CYP1A2_pIC50_TDI_condition,CYP2C9_pIC50_TDI_condition,CYP2D6_pIC50_TDI_condition,CYP3A4_pIC50_TDI_condition,CYP1A2_pIC50_TDI_condition_conf_high,CYP2C9_pIC50_TDI_condition_conf_high,...,CYP2D6_pIC50_direct_inhibition_conf_high,CYP3A4_pIC50_direct_inhibition_conf_high,CYP1A2_pIC50_direct_inhibition_conf_low,CYP2C9_pIC50_direct_inhibition_conf_low,CYP2D6_pIC50_direct_inhibition_conf_low,CYP3A4_pIC50_direct_inhibition_conf_low,CYP1A2_pIC50_direct_inhibition_std,CYP2C9_pIC50_direct_inhibition_std,CYP2D6_pIC50_direct_inhibition_std,CYP3A4_pIC50_direct_inhibition_std
0,OCNT-0000422,CC1=C(O)C(C=O)=C(CO)C=N1,NaN,False,NaN,NaN,NaN,2.004112,NaN,NaN,...,NaN,3.452393,NaN,NaN,NaN,1.048337,NaN,NaN,NaN,0.696258
1,OCNT-0001882,CC(C)(OC1=CC=C(Cl)C=C1)C(=O)O,NaN,NaN,NaN,2.026652,NaN,NaN,NaN,3.196343,...,NaN,NaN,NaN,1.069973,NaN,NaN,NaN,0.640165,NaN,NaN
2,OCNT-0007477,CCOC(=O)N1CSCC1C(=O)O,NaN,False,NaN,NaN,NaN,2.213487,NaN,NaN,...,NaN,3.717840,NaN,NaN,NaN,1.140199,NaN,NaN,NaN,0.720146
3,OCNT-0010068,O=C(O)C1=CN=CC=C1,NaN,False,NaN,NaN,NaN,1.975728,NaN,NaN,...,NaN,3.566120,NaN,NaN,NaN,1.073916,NaN,NaN,NaN,0.712205
4,OCNT-0014841,NCC1=CC=C(S(N)(=O)=O)C=C1,NaN,False,NaN,NaN,NaN,2.120996,NaN,NaN,...,NaN,3.429701,NaN,NaN,NaN,1.031887,NaN,NaN,NaN,0.687328


In [46]:
df = pd.read_csv(r"..\data\raw\cyp-challenge-TRAIN_inhibition.csv")

In [47]:
df.head()

,Molecule_Name,SMILES,CYP1A2_pIC50_direct_inhibition,CYP2C9_pIC50_direct_inhibition,CYP2D6_pIC50_direct_inhibition,CYP3A4_pIC50_direct_inhibition,CYP1A2_pIC50_direct_inhibition_conf_high,CYP2C9_pIC50_direct_inhibition_conf_high,CYP2D6_pIC50_direct_inhibition_conf_high,CYP3A4_pIC50_direct_inhibition_conf_high,CYP1A2_pIC50_direct_inhibition_conf_low,CYP2C9_pIC50_direct_inhibition_conf_low,CYP2D6_pIC50_direct_inhibition_conf_low,CYP3A4_pIC50_direct_inhibition_conf_low,CYP1A2_pIC50_direct_inhibition_std,CYP2C9_pIC50_direct_inhibition_std,CYP2D6_pIC50_direct_inhibition_std,CYP3A4_pIC50_direct_inhibition_std
0,OCNT-0000422,CC1=C(O)C(C=O)=C(CO)C=N1,NaN,NaN,NaN,2.101240,NaN,NaN,NaN,3.452393,NaN,NaN,NaN,1.048337,NaN,NaN,NaN,0.696258
1,OCNT-0001882,CC(C)(OC1=CC=C(Cl)C=C1)C(=O)O,NaN,2.169687,NaN,NaN,NaN,3.226921,NaN,NaN,NaN,1.069973,NaN,NaN,NaN,0.640165,NaN,NaN
2,OCNT-0007477,CCOC(=O)N1CSCC1C(=O)O,NaN,NaN,NaN,2.577796,NaN,NaN,NaN,3.717840,NaN,NaN,NaN,1.140199,NaN,NaN,NaN,0.720146
3,OCNT-0010068,O=C(O)C1=CN=CC=C1,NaN,NaN,NaN,2.273187,NaN,NaN,NaN,3.566120,NaN,NaN,NaN,1.073916,NaN,NaN,NaN,0.712205
4,OCNT-0014841,NCC1=CC=C(S(N)(=O)=O)C=C1,NaN,NaN,NaN,2.054066,NaN,NaN,NaN,3.429701,NaN,NaN,NaN,1.031887,NaN,NaN,NaN,0.687328


In [48]:
df[['CYP1A2_pIC50_direct_inhibition',
    'CYP2C9_pIC50_direct_inhibition',
    'CYP2D6_pIC50_direct_inhibition',
    'CYP3A4_pIC50_direct_inhibition']].count()

CYP1A2_pIC50_direct_inhibition    1412
CYP2C9_pIC50_direct_inhibition    1285
CYP2D6_pIC50_direct_inhibition    1493
CYP3A4_pIC50_direct_inhibition    2335
dtype: int64

In [49]:
def generate_morgan_fingerprint(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=radius,
        nBits=n_bits
    )

    return np.array(fp)

df['morgan_fingerprint'] = df['SMILES'].apply(generate_morgan_fingerprint)

[23:22:01] DEPRECATION WARNING: please use MorganGenerator
[23:22:01] DEPRECATION WARNING: please use MorganGenerator
[23:22:01] DEPRECATION WARNING: please use MorganGenerator
[23:22:01] DEPRECATION WARNING: please use MorganGenerator
[23:22:01] DEPRECATION WARNING: please use MorganGenerator
[23:22:01] DEPRECATION WARNING: please use MorganGenerator
[23:22:01] DEPRECATION WARNING: please use MorganGenerator
[23:22:01] DEPRECATION WARNING: please use MorganGenerator
[23:22:01] DEPRECATION WARNING: please use MorganGenerator
[23:22:01] DEPRECATION WARNING: please use MorganGenerator
[23:22:01] DEPRECATION WARNING: please use MorganGenerator
[23:22:01] DEPRECATION WARNING: please use MorganGenerator
[23:22:01] DEPRECATION WARNING: please use MorganGenerator
[23:22:01] DEPRECATION WARNING: please use MorganGenerator
[23:22:01] DEPRECATION WARNING: please use MorganGenerator
[23:22:01] DEPRECATION WARNING: please use MorganGenerator
[23:22:01] DEPRECATION WARNING: please use MorganGenerat

In [ ]:
cols = ['CYP1A2_pIC50_direct_inhibition','CYP2C9_pIC50_direct_inhibition','CYP2D6_pIC50_direct_inhibition','CYP3A4_pIC50_direct_inhibition']

# 5-fold CV
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

models = {"XGBoost": XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42 ),
          "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1 ),
          "SVR": SVR(kernel='rbf', C=1.0, epsilon=0.1)}


for col in cols:

    # Keep only molecules with a label for this endpoint
    tmp_df = df[[col, 'morgan_fingerprint']].dropna(subset=[col])

    X = np.array(list(tmp_df['morgan_fingerprint']))
    y = tmp_df[col].values

    print(f"\n{'=' * 60}")
    print(f"Target: {col}")
    print(f"Samples: {len(y)}")
    print(f"{'=' * 60}")

    for name, model in models.items():

        scores = cross_validate(
            model,
            X, y, cv=kf,
            scoring={'R2': 'r2',
                'MSE': 'neg_mean_squared_error',
                'RMSE': 'neg_root_mean_squared_error'
            },
            n_jobs=-1)

        r2 = scores['test_R2']
        mse = -scores['test_MSE']
        rmse = -scores['test_RMSE']

        print(f"\n{name}")

        print(f"R²   : {r2.mean():.4f} ± {r2.std():.4f}")
        print(f"MSE  : {mse.mean():.4f} ± {mse.std():.4f}")
        print(f"RMSE : {rmse.mean():.4f} ± {rmse.std():.4f}")


Target: CYP1A2_pIC50_direct_inhibition
Samples: 1412

XGBoost
R²   : 0.1335 ± 0.0200
MSE  : 0.9193 ± 0.1006
RMSE : 0.9574 ± 0.0507

Random Forest
R²   : 0.1252 ± 0.0251
MSE  : 0.9276 ± 0.0977
RMSE : 0.9619 ± 0.0491

SVR
R²   : 0.1852 ± 0.0248
MSE  : 0.8634 ± 0.0857
RMSE : 0.9281 ± 0.0447

Target: CYP2C9_pIC50_direct_inhibition
Samples: 1285

XGBoost
R²   : 0.1008 ± 0.0440
MSE  : 0.5485 ± 0.0469
RMSE : 0.7400 ± 0.0311

Random Forest
R²   : 0.0676 ± 0.0245
MSE  : 0.5690 ± 0.0445
RMSE : 0.7538 ± 0.0293

SVR
R²   : 0.1895 ± 0.0270
MSE  : 0.4946 ± 0.0408
RMSE : 0.7027 ± 0.0289

Target: CYP2D6_pIC50_direct_inhibition
Samples: 1493

XGBoost
R²   : 0.0627 ± 0.0283
MSE  : 0.7843 ± 0.0450
RMSE : 0.8853 ± 0.0254

Random Forest
R²   : 0.0805 ± 0.0147
MSE  : 0.7696 ± 0.0436
RMSE : 0.8769 ± 0.0247

SVR
R²   : 0.1247 ± 0.0150
MSE  : 0.7327 ± 0.0435
RMSE : 0.8556 ± 0.0253

Target: CYP3A4_pIC50_direct_inhibition
Samples: 2335

XGBoost
R²   : 0.3275 ± 0.0277
MSE  : 0.7998 ± 0.0276
RMSE : 0.8942 ± 0.015